In [1]:
print("hello")

hello


In [2]:
print("all ok")

all ok


In [3]:
import uuid
from dataclasses import dataclass
from langgraph.graph import (
    StateGraph,
    MessagesState,
    START,
    END
)
from langgraph.runtime import Runtime

In [5]:
from langchain.chat_models import init_chat_model
model = init_chat_model(
    "openai:gpt-5-nano",
    temperature=0
)

In [6]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

In [7]:
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()

In [8]:
@dataclass
class Context:
    user_id: str


In [9]:
Context.user_id=str(uuid.uuid4())

In [10]:
Context.user_id

'c79ec0a1-4021-4704-bfc3-61b80e1d6b06'

In [11]:
def assistant_node(
    state: MessagesState,
    runtime: Runtime[Context]
):
    user_id = runtime.context.user_id
    namespace = ("users", user_id, "memories")
    user_message = state["messages"][-1].content
    
    if user_message.lower().startswith("remember:"):
        memory = user_message[len("remember:"):].strip()
        runtime.store.put(
            namespace,
            str(uuid.uuid4()),
            {
                "data": memory
            }
        )
        print(
            f"[MEMORY SAVED]: {memory}"
        )
        
    memories = runtime.store.search(namespace)
    memory_text = "\n".join(
        memory.value["data"]
        for memory in memories
    )
    if not memory_text:
        memory_text = "No saved memories."
        
    
    system_prompt = f"""
    You are a helpful assistant.

    Long-term memories about this user:

    {memory_text}
    """


    response = model.invoke(
        [
            {
                "role": "system",
                "content": system_prompt
            },
            *state["messages"]
        ]
    )
    return {
        "messages": [response]
    }

In [12]:
builder = StateGraph(
    MessagesState,
    context_schema=Context
)


In [13]:
builder.add_node(
    "assistant",
    assistant_node
)

In [14]:
builder.add_edge(
    START,
    "assistant"
)

builder.add_edge(
    "assistant",
    END
)

In [15]:
graph = builder.compile(
    checkpointer=checkpointer,
    store=store
)

In [16]:
def chat(
    message,
    user_id,
    thread_id
):

    result = graph.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": message
                }
            ]
        },

        config={
            "configurable": {
                "thread_id": thread_id
            }
        },

        context=Context(
            user_id=user_id
        )
    )


    answer = result["messages"][-1].content

    print("\nUSER:")
    print(message)

    print("\nASSISTANT:")
    print(answer)

    print("\n" + "=" * 60)

In [17]:
chat(
    message="My current project is Agentic RAG.",
    user_id="sudheer_G",
    thread_id="thread_1"
)


USER:
My current project is Agentic RAG.

ASSISTANT:
Nice project. Do you want help planning, designing, or coding? I can tailor this to your needs. A few quick questions to start would help, but here are concrete options you can pick from:

Clarifying questions
- What are the main use cases for Agentic RAG (e.g., document QA, decision support, tool execution)?
- Fully autonomous agent or semi-autonomous with human-in-the-loop?
- Target latency, scale, and safety constraints (privacy, hallucinations, tool misuse)?
- Preferred tech stack or constraints (language, LLM provider, vector store, deployment environment)?

Start-to-finish guidance ( MVP-oriented)
- Architecture overview
  - Ingest: collect data sources (docs, PDFs, web data).
  - Embedding & storage: embed text and store in a vector DB (e.g., Pinecone, Milvus, Chroma).
  - Retrieval: a retriever that fetches relevant chunks for a query.
  - Reasoning/Generation: an LLM that composes a plan, answers, or actions.
  - Agent laye

In [18]:
chat(
    message="What is my current project?",
    user_id="sudheer_G",
    thread_id="thread_1"
)


USER:
What is my current project?

ASSISTANT:
Your current project is Agentic RAG.

Brief description:
- A framework to build an AI agent that uses retrieval-augmented generation. It ingests data (docs, PDFs, web data), embeds and stores it in a vector store, retrieves relevant chunks, and uses an LLM to plan, reason, and generate answers or actions. An agent/orchestrator decides the next steps (answer, fetch more data, or execute tools), with optional safety guards and human-in-the-loop.

If you’d like, I can help with:
- A tailored one-page architecture spec for Agentic RAG
- A concrete MVP plan with milestones and timeline
- A starter code skeleton (folders, modules, prompts) to get going
- Evaluation and safety prompts/checks

What would you like me to prepare next?



In [19]:
chat(
    message="Remember: my favorite programming language is Python.",
    user_id="sudheer_G",
    thread_id="thread_1"
)

[MEMORY SAVED]: my favorite programming language is Python.

USER:
Remember: my favorite programming language is Python.

ASSISTANT:
Noted. I’ll tailor all future guidance and code examples to Python, since that’s your favorite language.

If you’d like, I can jump straight to Python-first deliverables. Options (pick what you want):
- A Python-focused one-page architecture spec for Agentic RAG
- A concrete MVP plan with milestones and a rough timeline (Python stack)
- A starter Python code skeleton (folders, modules, prompts) you can run locally
- A set of evaluation and safety prompts tailored for Python implementations

Here’s a quick Python-centric starter outline you can reuse right away:

Project layout (Python-friendly)
- data/, embeddings/, vectors/  (data and index storage)
- src/
  - agent.py       (planner/loop / orchestrator)
  - retriever.py   (top-k retrieval from vector store)
  - prompt_utils.py (prompts for planning, answering, citing sources)
  - tools.py       (optiona

In [20]:
chat(
    message="What is my favorite programming language?",
    user_id="sudheer_G",
    thread_id="thread_2"
)


USER:
What is my favorite programming language?

ASSISTANT:
Python.



                 maha_123
                     ↓
              InMemoryStore
                     ↓
      "favorite language = Python"
                     ↓
             ┌───────┴───────┐
             ↓               ↓
         thread_1         thread_2

In [21]:
chat(
    message="Remember: my favorite language is Python.",
    user_id="sudheer_G",
    thread_id="sudheer_thread_1"
)

[MEMORY SAVED]: my favorite language is Python.

USER:
Remember: my favorite language is Python.

ASSISTANT:
Noted. Your favorite language is Python. I’ll tailor examples and explanations in Python when helpful. If you want me to remember other details, just tell me.



In [22]:
chat(
    message="Remember: my favorite language is Java.",
    user_id="rahul_456",
    thread_id="rahul_thread_1"
)

[MEMORY SAVED]: my favorite language is Java.

USER:
Remember: my favorite language is Java.

ASSISTANT:
Noted: your favorite language is Java. I’ll tailor Java-focused examples and explanations when relevant. Want me to remember any other preferences or provide a quick Java tip?



users
│
├── maha_123
│      └── memories
│           └── favorite language = Python
│
└── rahul_456
       └── memories
            └── favorite language = Java

In [23]:
chat(
    message="What is my favorite programming language?",
    user_id="rahul_456",
    thread_id="rahul_thread_2"
)


USER:
What is my favorite programming language?

ASSISTANT:
Java. Would you like to dive into Java topics or discuss anything specific about it?



In [24]:
chat(
    message="What is my favorite programming language?",
    user_id="sudheer_G",
    thread_id="sudheer_thread_2"
)


USER:
What is my favorite programming language?

ASSISTANT:
Python.



In [25]:
user_id = "sudheer_G"

In [26]:
namespace = (
    "users",
    user_id,
    "memories"
)


In [27]:
memories = store.search(namespace)

for memory in memories:
    print(memory.value)

{'data': 'my favorite programming language is Python.'}
{'data': 'my favorite language is Python.'}


In [28]:
memories = store.search(
    ("users", "sudheer_G", "memories")
)

for memory in memories:
    print(memory.value["data"])

my favorite programming language is Python.
my favorite language is Python.
